# When can a student pilot fly from Saint-Cyr?

15 years of METARs at Toussus-le-Noble (reference for Saint-Cyr, which has no METAR), turned into a flying
climatology: best months and hours, reasons for cancellations, odds of a 2-hour lesson, a morning forecast,
and a map of the aerodromes around Paris.

The student minimums are defined in `src/flying.py` (`STUDENT = Minima(...)`): edit them to match your own.

In [ ]:
# Get the code: uploaded zip, GitHub repository, or upload prompt
import os, sys, glob
if not os.path.exists("src"):
    if os.path.exists("/content"):
        os.chdir("/content")
    if not os.path.exists("vfr-climatology/src"):
        zips = glob.glob("vfr-climatology*.zip")
        if zips:
            !unzip -q -o "{zips[0]}"
        else:
            !git clone -q https://github.com/AElSama/vfr-climatology || echo "Repo not found"
    if not os.path.exists("vfr-climatology/src"):
        from google.colab import files
        print("Upload vfr-climatology.zip")
        zips = list(files.upload().keys())
        !unzip -q -o "{zips[0]}"
    %cd vfr-climatology
sys.path.insert(0, ".")
SYNTHETIC = False

## 1. Run the full study

The first run downloads the METAR archives (about 90 yearly files, several minutes). They are cached in `data/`.

In [ ]:
!python run.py {"--synthetic" if SYNTHETIC else ""}

## 2. Results

In [ ]:
import json
r = json.load(open("results.json"))
f, fc = r["flyable_share"], r["forecast_afternoon_from_7am"]
print(f"Daylight hours within student minimums: {f['student_annual']:.0%}  (regulatory VFR: {f['vfr_annual']:.0%})")
print("By month:", {m: f'{v:.0%}' for m, v in f['student_by_month'].items()})
print("Weekend day with at least one flyable 2h slot:", {m: f'{v:.0%}' for m, v in r['p_saturday_or_sunday_has_slot'].items()})
print("Best 2h lesson start:", {m: f"{v['start']} ({v['p']:.0%})" for m, v in r['best_2h_slot_by_month'].items()})
print("Main reason to stay on the ground:", r["main_reason_by_month"])
print(f"7am -> 14h-16h forecast, test {fc['test_years']}: Brier skill {fc['brier_skill']:+.0%}, AUC {fc['auc']:.2f}")
print(f"Afternoon flyable after morning fog/mist: {fc['p_afternoon_ok_if_morning_fog_or_mist']:.0%}, otherwise {fc['p_afternoon_ok_otherwise']:.0%}")

In [ ]:
from IPython.display import Image, display
for name in ["flyable_month_hour", "no_fly_reasons", "lesson_slots", "vfr_vs_student", "forecast_reliability", "map_static"]:
    display(Image(f"figures/{name}.png"))

## 3. Interactive map

Click an aerodrome for its monthly profile.

In [ ]:
from IPython.display import HTML
HTML(f'<iframe srcdoc="{open("docs/index.html", encoding="utf-8").read().replace(chr(34), "&quot;")}" width="100%" height="600" style="border:0"></iframe>')

## 4. Download the results (Colab)

With real data, `run.py` has filled the results section of `README.md`. This downloads the project with figures and map, ready for GitHub.

In [ ]:
!cd .. && zip -qr vfr-climatology-results.zip vfr-climatology -x "*/.git/*" "*/data/*" "*/__pycache__/*"
try:
    from google.colab import files
    files.download("../vfr-climatology-results.zip")
except ImportError:
    print("Not on Colab: the zip is in the parent folder.")